# LangGraph Advanced: Memory, Interrupts & Streaming

This notebook builds on the previous tutorials to explore LangGraph's advanced features:

1. **Short-Term Memory** - Checkpointers for conversation persistence
2. **Human-in-the-Loop (HITL)** - Interrupts for human approval/input
3. **Long-Term Memory** - Stores for cross-session memory
4. **Semantic Retrieval** - Vector search over memories
5. **Streaming** - Real-time updates from graph execution

---

In [14]:
import os
import uuid
from dotenv import load_dotenv
from typing import Literal, Annotated

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv()

model = ChatOpenAI(model="gpt-5-mini", temperature=0)

---
# Part 1: Short-Term Memory (Checkpointer)

**Checkpointers** save snapshots of the graph state at each super-step. This enables:
- **Conversation persistence** - Resume conversations across invocations
- **Time-travel debugging** - Replay from any checkpoint
- **Fault tolerance** - Recover from failures mid-execution

### How It Works
- Checkpoints are saved automatically at each super-step
- Each **thread** (identified by `thread_id`) maintains its own checkpoint history
- For production, use `PostgresSaver`, `RedisSaver`, etc. instead of `InMemorySaver`

### Key Methods
| Method | Purpose |
|--------|--------|
| `get_state(config)` | Get the latest state snapshot |
| `get_state_history(config)` | Get all checkpoints for a thread |
| `update_state(config, values)` | Manually update state (for forking) |

### Building a ReAct Agent with Checkpointing

This cell builds the same ReAct agent from Tutorial 02, but now we compile it with a **checkpointer**. The key addition is `builder.compile(checkpointer=checkpointer)` which enables state persistence across invocations.

In [15]:
from langgraph.checkpoint.memory import InMemorySaver

# Define tools
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"The weather in {city} is 72°F and sunny."

@tool  
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

tools = [get_weather, add_numbers]
model_with_tools = model.bind_tools(tools)

# Define state
class MessagesState(BaseModel):
    messages: Annotated[list[AnyMessage], add_messages] = Field(default_factory=list)

# Define nodes
def llm_call(state: MessagesState):
    return {
        "messages": [model_with_tools.invoke(
            [SystemMessage(content="You are a helpful assistant.")]
            + state.messages
        )]
    }

tool_node = ToolNode(tools)

def should_continue(state: MessagesState) -> Literal["tool_node", "__end__"]:
    if state.messages[-1].tool_calls:
        return "tool_node"
    return END

# Build graph
builder = StateGraph(MessagesState)
builder.add_node("llm_call", llm_call)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_call")
builder.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
builder.add_edge("tool_node", "llm_call")

# Compile WITH checkpointer for memory
checkpointer = InMemorySaver()
agent = builder.compile(checkpointer=checkpointer)

print("Agent compiled with checkpointer")

Agent compiled with checkpointer


### Visualize the Graph

Render the compiled graph as a mermaid diagram. This shows the flow: `START` → `llm_call` → (conditional) → `tool_node` or `END`.

In [16]:
from IPython.display import IFrame
import re
import base64
import urllib.parse

def display_mermaid(graph):
    """Display LangGraph as Mermaid diagram via mermaid.live embed."""
    mermaid_syntax = graph.get_graph().draw_mermaid()
    # Remove <p> tags that LangGraph 1.0.5+ adds
    mermaid_syntax = re.sub(r'<p>|</p>', '', mermaid_syntax)
    
    # Encode for mermaid.live URL
    graph_def = base64.urlsafe_b64encode(mermaid_syntax.encode()).decode()
    
    # Create standalone HTML with mermaid
    html_content = f"""<!DOCTYPE html>
<html><head>
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
</head><body>
<pre class="mermaid">{mermaid_syntax}</pre>
<script>mermaid.initialize({{startOnLoad:true}});</script>
</body></html>"""
    
    # Encode as data URL
    data_url = "data:text/html;base64," + base64.b64encode(html_content.encode()).decode()
    return IFrame(data_url, width=600, height=400)

display_mermaid(agent)

### Demo: Conversation Memory

With a checkpointer, each `thread_id` gets its own conversation history. The agent will remember what was said in previous turns within the same thread.

In [17]:
# Each thread maintains its own conversation history
# The thread_id is REQUIRED when using a checkpointer

config = {"configurable": {"thread_id": "user-123"}}

# First message
result = agent.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Jacob.")]},
    config=config
)
print("Turn 1:")
result["messages"][-1].pretty_print()

Turn 1:
================================== Ai Message ==================================

Hi Jacob — nice to meet you! How can I help today?


In [18]:
# Second message - the agent remembers the first message!
result = agent.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    config=config  # Same thread_id
)
print("Turn 2:")
result["messages"][-1].pretty_print()

Turn 2:
================================== Ai Message ==================================

Your name is Jacob.


The agent remembers the name from the previous turn because we used the same `thread_id`. Try changing the thread_id to see that different threads have separate memories.

### Inspecting Checkpoint History

Every super-step creates a checkpoint. Use `get_state_history()` to see all snapshots for a thread. Each checkpoint has a unique `checkpoint_id` you can use for time-travel.

In [19]:
# View all checkpoints (snapshots) for this thread
print("Checkpoint History:")
for i, state in enumerate(agent.get_state_history(config)):
    msg_count = len(state.values.get("messages", []))
    print(f"  [{i}] checkpoint_id={state.config['configurable']['checkpoint_id'][:8]}... messages={msg_count}")

Checkpoint History:
  [0] checkpoint_id=1f0e4c2d... messages=4
  [1] checkpoint_id=1f0e4c2d... messages=3
  [2] checkpoint_id=1f0e4c2d... messages=2
  [3] checkpoint_id=1f0e4c2d... messages=2
  [4] checkpoint_id=1f0e4c2d... messages=1
  [5] checkpoint_id=1f0e4c2d... messages=0


### Time-Travel: Forking from a Previous State

You can "go back in time" by specifying a `checkpoint_id`. This creates a **fork** - the original history is preserved, and you start a new branch from that point.

In [20]:
# Time-Travel: Replay from an earlier checkpoint
# This creates a FORK - original history is preserved

history = list(agent.get_state_history(config))
earlier_checkpoint = history[-2]  # Second-to-last state

# Fork from that checkpoint with a different message
fork_config = {
    "configurable": {
        "thread_id": "user-123",
        "checkpoint_id": earlier_checkpoint.config["configurable"]["checkpoint_id"]
    }
}

# Resume from the earlier state
result = agent.invoke(
    {"messages": [HumanMessage(content="Actually, call me Bob instead.")]},
    config=fork_config
)
print("Forked conversation:")
result["messages"][-1].pretty_print()

Forked conversation:
================================== Ai Message ==================================

Nice to meet you, Bob! How can I help you today?


---
# Part 2: Human-in-the-Loop (HITL) Interrupts

**Interrupts** pause graph execution to get human input. The graph state is saved, and execution resumes when you provide a response.

### How It Works
1. Graph execution reaches an `interrupt()` call
2. State is saved via the checkpointer
3. The interrupt value is returned under `__interrupt__`
4. You resume with `Command(resume=<value>)`, which becomes the `interrupt()` return value

### Use Cases
- **Approval workflows** - Human approves/rejects an action
- **Content review** - Human edits generated content
- **Tool approval** - Human approves sensitive tool calls (e.g., sending emails)
- **Input validation** - Loop until valid input is provided

> **Warning**: Don't put `interrupt()` inside try/except blocks, and don't let the order of interrupts change dynamically.

### Building an Approval Workflow

This example creates a workflow where:
1. `propose` node suggests an action
2. `approval_gate` pauses with `interrupt()` for human decision
3. Based on the response, routes to `execute` or `cancel`

The `interrupt()` call pauses the graph and returns a payload to the caller. When resumed with `Command(resume=...)`, the value becomes the return value of `interrupt()`.

In [21]:
from langgraph.types import Command, interrupt

# Example: Approval Workflow
# The agent proposes an action, human approves or rejects

class ApprovalState(BaseModel):
    action: str = ""
    status: str = "pending"

def propose_action(state: ApprovalState):
    """Agent proposes an action."""
    return {"action": "Send email to alice@example.com"}

def approval_gate(state: ApprovalState) -> Command[Literal["execute", "cancel"]]:
    """Pause for human approval."""
    
    # interrupt() pauses execution and returns this value to the caller
    # The value can be any JSON-serializable data to display in a UI
    decision = interrupt({
        "question": "Approve this action?",
        "proposed_action": state.action,
    })
    
    # When resumed, 'decision' contains whatever was passed to Command(resume=...)
    if decision.get("approved"):
        return Command(goto="execute")
    return Command(goto="cancel")

def execute_action(state: ApprovalState):
    return {"status": "executed"}

def cancel_action(state: ApprovalState):
    return {"status": "cancelled"}

# Build approval workflow
approval_builder = StateGraph(ApprovalState)
approval_builder.add_node("propose", propose_action)
approval_builder.add_node("approval_gate", approval_gate)
approval_builder.add_node("execute", execute_action)
approval_builder.add_node("cancel", cancel_action)

approval_builder.add_edge(START, "propose")
approval_builder.add_edge("propose", "approval_gate")
approval_builder.add_edge("execute", END)
approval_builder.add_edge("cancel", END)

approval_checkpointer = InMemorySaver()
approval_graph = approval_builder.compile(checkpointer=approval_checkpointer)

print("Approval workflow compiled")

Approval workflow compiled


### Starting the Workflow

When we invoke the graph, it runs until it hits the `interrupt()`. The result contains `__interrupt__` with the payload we passed to `interrupt()`. The graph is now paused, waiting for us to resume.

In [22]:
# Start the workflow - it will pause at the interrupt
config = {"configurable": {"thread_id": "approval-001"}}

result = approval_graph.invoke({"action": "", "status": "pending"}, config=config)

# The graph paused at the interrupt
print("Graph paused. Interrupt payload:")
print(result["__interrupt__"])

Graph paused. Interrupt payload:
[Interrupt(value={'question': 'Approve this action?', 'proposed_action': 'Send email to alice@example.com'}, id='423183213a2d82ddecb3164f85ec5fbd')]


### Resuming with Approval

To resume, we call `invoke()` with `Command(resume=...)`. The value we pass becomes the return value of `interrupt()` in the paused node. Here we approve, so the graph routes to `execute`.

In [23]:
# Resume with approval
result = approval_graph.invoke(
    Command(resume={"approved": True}),
    config=config
)

print(f"Final status: {result['status']}")

Final status: executed


### Resuming with Rejection

Same workflow, but this time we reject. The graph routes to `cancel` instead. Note we use a different `thread_id` to start fresh.

In [24]:
# Try again with rejection
config2 = {"configurable": {"thread_id": "approval-002"}}

# Start workflow
result = approval_graph.invoke({"action": "", "status": "pending"}, config=config2)
print("Interrupt:", result["__interrupt__"])

# Resume with rejection
result = approval_graph.invoke(
    Command(resume={"approved": False}),
    config=config2
)
print(f"Final status: {result['status']}")

Interrupt: [Interrupt(value={'question': 'Approve this action?', 'proposed_action': 'Send email to alice@example.com'}, id='d9c24060eca14aa6c921587d811435c6')]
Final status: cancelled


### Debug Interrupts

You can set interrupts at compile time for debugging without modifying node code.
This is useful for inspecting state at specific points in the graph.

In [25]:
# Compile with debug interrupts - pause before/after specific nodes
debug_graph = builder.compile(
    checkpointer=InMemorySaver(),
    interrupt_before=["tool_node"],  # Pause before tool execution
    # interrupt_after=["llm_call"],  # Could also pause after LLM
)

config = {"configurable": {"thread_id": "debug-001"}}

# This will pause before tool_node
result = debug_graph.invoke(
    {"messages": [HumanMessage(content="What's the weather in Paris?")]},
    config=config
)

# Check current state
state = debug_graph.get_state(config)
print("Paused at:", state.next)  # Shows which node is next
print("Pending tool call:", state.values["messages"][-1].tool_calls)

Paused at: ('tool_node',)
Pending tool call: [{'name': 'get_weather', 'args': {'city': 'Paris'}, 'id': 'call_SZlw0YAkcQ3lydLQVUnyD7gN', 'type': 'tool_call'}]


Compile with `interrupt_before=["tool_node"]` to pause before tool execution. This lets you inspect pending tool calls before they run - useful for debugging or adding approval for sensitive tools.

In [26]:
# Resume execution (pass None to continue from current state)
result = debug_graph.invoke(None, config=config)
print("Resumed and completed:")
result["messages"][-1].pretty_print()

Resumed and completed:
================================== Ai Message ==================================

It's 72°F and sunny in Paris.


Pass `None` as input to resume from the current state. The graph continues from where it paused.

---
# Part 3: Long-Term Memory (Store)

**Stores** provide persistent memory that spans across threads. Unlike checkpointers (which are per-thread), stores let you share information across conversations.

### Key Concepts
- **Namespace**: A tuple that organizes memories, e.g., `(user_id, "preferences")`
- **put()**: Save a memory with a unique key
- **search()**: Retrieve memories (optionally with semantic search)
- **get()**: Retrieve a specific memory by key

### Use Cases
- User preferences that persist across sessions
- Facts learned about a user over time
- Agent self-improvement via stored learnings

### Production Options
| Store Type | Use Case |
|------------|----------|
| `InMemoryStore` | Development/testing |
| `PostgresStore` | Production with SQL |
| `RedisStore` | Fast key-value access |

### Basic Store Operations

Stores use **namespaces** (tuples) to organize data. Here we create a namespace for Alice's preferences and demonstrate `put()` to save and `search()` to retrieve memories.

In [27]:
from langgraph.store.memory import InMemoryStore

# Create an in-memory store
store = InMemoryStore()

# Stores are organized by namespace (a tuple)
user_id = "alice"
namespace = (user_id, "preferences")

# Save some memories
store.put(namespace, "food", {"preference": "vegetarian", "favorite": "pasta"})
store.put(namespace, "communication", {"style": "concise", "emoji": False})

# Retrieve all memories in a namespace
memories = store.search(namespace)
print("Stored memories:")
for mem in memories:
    print(f"  {mem.key}: {mem.value}")

Stored memories:
  food: {'preference': 'vegetarian', 'favorite': 'pasta'}
  communication: {'style': 'concise', 'emoji': False}


Use `get()` when you know the exact key. Use `search()` to retrieve all items in a namespace or to do semantic search (when embeddings are configured).

In [28]:
# Get a specific memory by key
food_pref = store.get(namespace, "food")
print(f"Food preference: {food_pref.value}")

Food preference: {'preference': 'vegetarian', 'favorite': 'pasta'}


### Using Store in a Graph

Nodes can access the store by adding `store: BaseStore` as a parameter.
LangGraph injects the store automatically at runtime.

In [29]:
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore

# Create a fresh store for this example
memory_store = InMemoryStore()

class ChatState(BaseModel):
    messages: Annotated[list[AnyMessage], add_messages] = Field(default_factory=list)

def chat_with_memory(state: ChatState, config: RunnableConfig, *, store: BaseStore):
    """Chat node that reads and writes to long-term memory."""
    
    # Get user_id from config
    user_id = config["configurable"].get("user_id", "default")
    namespace = (user_id, "facts")
    
    # Retrieve existing memories
    memories = store.search(namespace, limit=5)
    memory_context = "\n".join([f"- {m.value['fact']}" for m in memories]) or "No memories yet."
    
    # Build system prompt with memories
    system = f"""You are a helpful assistant with memory of past conversations.
    
Known facts about the user:
{memory_context}

If the user shares new facts about themselves, acknowledge them naturally."""
    
    # Generate response
    response = model.invoke(
        [SystemMessage(content=system)] + state.messages
    )
    
    # Simple extraction: If user says "I like X" or "I am X", store it
    last_user_msg = state.messages[-1].content.lower()
    if "i like" in last_user_msg or "i am" in last_user_msg or "my name is" in last_user_msg:
        memory_id = str(uuid.uuid4())
        store.put(namespace, memory_id, {"fact": state.messages[-1].content})
    
    return {"messages": [response]}

# Build graph with store
memory_builder = StateGraph(ChatState)
memory_builder.add_node("chat", chat_with_memory)
memory_builder.add_edge(START, "chat")
memory_builder.add_edge("chat", END)

memory_agent = memory_builder.compile(
    checkpointer=InMemorySaver(),
    store=memory_store  # Pass the store at compile time
)

print("Agent compiled with long-term memory store")

Agent compiled with long-term memory store


In [30]:
# Visualize the memory agent graph
display_mermaid(memory_agent)

### How Memory Storage Works Under the Hood

When Bob sends a message, here's what happens step by step:

**1. Config specifies the user identity:**
```python
config = {"configurable": {"thread_id": "thread-1", "user_id": "bob"}}
```
- `thread_id` → Used by the **checkpointer** (short-term memory within a conversation)
- `user_id` → Used by the **store** (long-term memory across conversations)

**2. Node receives injected dependencies:**
```python
def chat_with_memory(state: ChatState, config: RunnableConfig, *, store: BaseStore):
```
- LangGraph automatically injects `config` and `store` when the node runs
- The `store` parameter is a keyword-only argument (after `*`)

**3. Namespace organizes memories:**
```python
user_id = config["configurable"].get("user_id", "default")  # "bob"
namespace = (user_id, "facts")  # ("bob", "facts")
```
- Namespaces are tuples that create a hierarchy
- All of Bob's facts live under `("bob", "facts")`
- Different users have isolated namespaces

**4. Memory is stored with put():**
```python
memory_id = str(uuid.uuid4())  # Random unique key
store.put(namespace, memory_id, {"fact": "Hi! My name is Bob and I like pizza."})
```
- `namespace` = `("bob", "facts")` → Where to store
- `memory_id` = UUID → Unique identifier for this memory
- `{"fact": ...}` → The actual data (any JSON-serializable dict)

**5. Memories are retrieved with search():**
```python
memories = store.search(namespace, limit=5)
```
- Returns all items in the namespace (up to limit)
- When semantic search is enabled, you can pass `query="..."` for vector similarity

### Memory Structure Diagram

```
InMemoryStore
├── ("bob", "facts")
│   ├── "abc-123..." → {"fact": "My name is Bob and I like pizza."}
│   └── "def-456..." → {"fact": "I am a software engineer."}
├── ("alice", "facts")
│   └── "ghi-789..." → {"fact": "I like hiking."}
└── ("bob", "preferences")
    └── "jkl-012..." → {"style": "casual"}
```

The key insight: **checkpointer** gives memory within a thread, **store** gives memory across threads for the same user.

In [31]:
# Thread 1: Share some facts
config = {"configurable": {"thread_id": "thread-1", "user_id": "bob"}}

result = memory_agent.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Bob and I like pizza.")]},
    config=config
)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hi Bob — nice to meet you! I’ll remember that you like pizza. What’s your favorite kind or toppings? Want recommendations, recipes, or nearby places that make great pizza?


### Cross-Thread Memory Persistence

Now we use a **different thread** but the **same user_id**. The store remembers Bob's facts even though this is a new conversation thread. This is the power of long-term memory!

In [32]:
# Thread 2: Different thread, same user - memories persist!
config2 = {"configurable": {"thread_id": "thread-2", "user_id": "bob"}}

result = memory_agent.invoke(
    {"messages": [HumanMessage(content="What do you know about me?")]},
    config=config2
)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

I know your name is Bob and that you like pizza. 

Would you like me to remember anything else (favorite toppings, dietary restrictions, where you live, hobbies, etc.), or update/forget anything?


### Direct Store Inspection

You can access the store directly (outside of a node) to inspect what's been saved. This is useful for debugging and understanding what the agent has learned.

In [33]:
# Inspect stored memories directly
print("\nStored memories for 'bob':")
for mem in memory_store.search(("bob", "facts")):
    print(f"  [{mem.key[:8]}...] {mem.value}")


Stored memories for 'bob':
  [09aa9e62...] {'fact': 'Hi! My name is Bob and I like pizza.'}


---
# Part 4: Semantic Retrieval Over Memory

For larger memory stores, you can enable **semantic search** using embeddings.
This allows natural language queries to find relevant memories.

### Configuration
| Parameter | Description |
|-----------|-------------|
| `embed` | The embedding model to use |
| `dims` | Embedding dimensions (e.g., 1536 for OpenAI) |
| `fields` | Which fields to embed (use `$` for entire value) |

### Selective Indexing
You can control which memories are embedded:
```python
# Only embed specific fields
store.put(ns, key, value, index=["field_to_embed"])

# Don't embed (still retrievable by key, not by search)
store.put(ns, key, value, index=False)
```

### Creating a Semantic Store

To enable vector search, pass an `index` config with:
- `embed`: The embedding model (LangChain Embeddings object)
- `dims`: Embedding dimensions (1536 for OpenAI's small model)
- `fields`: Which fields in your data to embed

We then seed it with sample memories to demonstrate search.

In [34]:
from langchain_openai import OpenAIEmbeddings

# Create store with semantic search enabled
semantic_store = InMemoryStore(
    index={
        "embed": OpenAIEmbeddings(model="text-embedding-3-small"),
        "dims": 1536,
        "fields": ["content"],  # Which fields to embed
    }
)

# Seed some memories
user_ns = ("user123", "memories")

memories_to_seed = [
    {"content": "User prefers vegetarian food and loves Italian cuisine"},
    {"content": "User works as a software engineer at a startup"},
    {"content": "User has a dog named Max and takes him to the park daily"},
    {"content": "User is learning to play guitar and practices on weekends"},
    {"content": "User allergic to peanuts - important dietary restriction"},
]

for mem in memories_to_seed:
    semantic_store.put(user_ns, str(uuid.uuid4()), mem)

print(f"Seeded {len(memories_to_seed)} memories with embeddings")

Seeded 5 memories with embeddings


### Semantic Search Demo

Pass a natural language `query` to `search()`. The store uses embeddings to find semantically similar memories, not just keyword matches. Notice how "food preferences" finds both the vegetarian preference AND the peanut allergy!

In [35]:
# Semantic search: Find relevant memories using natural language
results = semantic_store.search(
    user_ns,
    query="What are the user's food preferences?",
    limit=3
)

print("Query: 'What are the user's food preferences?'")
print("\nTop matches:")
for r in results:
    print(f"  - {r.value['content']}")

Query: 'What are the user's food preferences?'

Top matches:
  - User prefers vegetarian food and loves Italian cuisine
  - User allergic to peanuts - important dietary restriction
  - User has a dog named Max and takes him to the park daily


In [36]:
# Another semantic search
results = semantic_store.search(
    user_ns,
    query="hobbies and activities",
    limit=2
)

print("Query: 'hobbies and activities'")
print("\nTop matches:")
for r in results:
    print(f"  - {r.value['content']}")

Query: 'hobbies and activities'

Top matches:
  - User is learning to play guitar and practices on weekends
  - User prefers vegetarian food and loves Italian cuisine


---
# Part 5: Self-Improving Agent with Seeded Memory

Agents can store learnings from interactions and use them to improve over time.
This pattern combines:
- **Seeded memories** - Pre-populate with known best practices
- **Semantic search** - Retrieve relevant learnings for each query
- **Runtime learning** - Store new insights from interactions

### Pattern: Memory-Augmented Response
1. User sends a message
2. Search memory for relevant learnings
3. Inject learnings into the prompt
4. Generate response
5. (Optional) Extract and store new learnings

### Seeding Agent Learnings

We create a semantic store and pre-populate it with "best practices" the agent should follow. These learnings will be retrieved based on the user's query and injected into the prompt.

In [37]:
# Create store for agent learnings
learnings_store = InMemoryStore(
    index={
        "embed": OpenAIEmbeddings(model="text-embedding-3-small"),
        "dims": 1536,
        "fields": ["lesson", "context"],
    }
)

# Seed with initial learnings (best practices)
agent_ns = ("agent", "learnings")

initial_learnings = [
    {
        "lesson": "When users ask about weather, they usually want a brief summary, not detailed forecasts",
        "context": "weather queries",
        "source": "seed"
    },
    {
        "lesson": "Math calculations should show the steps, not just the final answer",
        "context": "math and calculations",
        "source": "seed"
    },
    {
        "lesson": "If a user seems frustrated, acknowledge their feelings before solving the problem",
        "context": "emotional interactions",
        "source": "seed"
    },
    {
        "lesson": "When explaining code, use simple analogies that relate to everyday concepts",
        "context": "code explanations",
        "source": "seed"
    },
]

for learning in initial_learnings:
    learnings_store.put(agent_ns, str(uuid.uuid4()), learning)

print(f"Seeded {len(initial_learnings)} agent learnings")

Seeded 4 agent learnings


### Building the Learning Agent

This agent has two nodes:
1. `retrieve` - Searches the learnings store for relevant best practices
2. `respond` - Generates a response with the learnings injected into the system prompt

The `retrieved_learnings` field in state shows which learnings were used.

In [38]:
class LearningAgentState(BaseModel):
    messages: Annotated[list[AnyMessage], add_messages] = Field(default_factory=list)
    retrieved_learnings: list[str] = Field(default_factory=list)

def retrieve_learnings(state: LearningAgentState, config: RunnableConfig, *, store: BaseStore):
    """Retrieve relevant learnings based on the user's message."""
    last_msg = state.messages[-1].content
    
    # Semantic search for relevant learnings
    results = store.search(
        ("agent", "learnings"),
        query=last_msg,
        limit=2
    )
    
    learnings = [r.value["lesson"] for r in results]
    return {"retrieved_learnings": learnings}

def respond_with_learnings(state: LearningAgentState):
    """Generate response using retrieved learnings."""
    learnings_context = "\n".join([f"- {l}" for l in state.retrieved_learnings])
    
    system = f"""You are a helpful assistant that learns from experience.

Apply these learnings to your response:
{learnings_context}

Be helpful and apply the learnings where relevant."""
    
    response = model.invoke([SystemMessage(content=system)] + state.messages)
    return {"messages": [response]}

# Build the learning agent
learning_builder = StateGraph(LearningAgentState)
learning_builder.add_node("retrieve", retrieve_learnings)
learning_builder.add_node("respond", respond_with_learnings)
learning_builder.add_edge(START, "retrieve")
learning_builder.add_edge("retrieve", "respond")
learning_builder.add_edge("respond", END)

learning_agent = learning_builder.compile(
    checkpointer=InMemorySaver(),
    store=learnings_store
)

print("Self-improving agent compiled")

Self-improving agent compiled


### Testing: Math Query

The agent retrieves the "show your steps" learning and applies it to the math question. Compare this response to a vanilla LLM that might just say "105".

In [39]:
# Test the agent - it should apply relevant learnings
config = {"configurable": {"thread_id": "learning-test-1"}}

result = learning_agent.invoke(
    {"messages": [HumanMessage(content="What's 15 times 7?")]},
    config=config
)

print("Retrieved learnings:", result["retrieved_learnings"])
print("\nResponse:")
result["messages"][-1].pretty_print()

Retrieved learnings: ['Math calculations should show the steps, not just the final answer', 'When explaining code, use simple analogies that relate to everyday concepts']

Response:
================================== Ai Message ==================================

Step 1: Break 15 into 10 + 5.
Step 2: Multiply each by 7: 10×7 = 70 and 5×7 = 35.
Step 3: Add the results: 70 + 35 = 105.

So, 15 × 7 = 105.


### Testing: Emotional Query

The agent retrieves the "acknowledge feelings" learning and applies it. Notice how the response starts with empathy before diving into solutions.

In [40]:
# Test with an emotional query
config = {"configurable": {"thread_id": "learning-test-2"}}

result = learning_agent.invoke(
    {"messages": [HumanMessage(content="I'm so frustrated! This code keeps breaking and I don't know why.")]},
    config=config
)

print("Retrieved learnings:", result["retrieved_learnings"])
print("\nResponse:")
result["messages"][-1].pretty_print()

Retrieved learnings: ['When explaining code, use simple analogies that relate to everyday concepts', 'If a user seems frustrated, acknowledge their feelings before solving the problem']

Response:
================================== Ai Message ==================================

I’m sorry you’re hitting that wall — that’s really frustrating. I can help you track it down. To start, tell me the language / framework you’re using and paste the code + error you see (see what to include below). If you’d rather I walk you through some quick checks first, try the checklist after that.

What I need from you to help fastest
- The exact error message and full stack trace (copy-paste, not paraphrased).  
- The small chunk of code that triggers it (or better: a minimal reproducible example).  
- What you expected to happen vs what happened.  
- Your runtime environment: OS, language and version (e.g., Python 3.11, Node 18, Java 17), framework versions, build tool.  
- Steps to reproduce (commands yo

---
# Part 6: Streaming

Streaming provides real-time updates as the graph executes.
Different modes give different levels of detail.

### Stream Modes

| Mode | Description | Use Case |
|------|-------------|----------|
| `values` | Full state snapshot after each node | Debugging, state inspection |
| `updates` | Only the changes from each node | Progress tracking |
| `messages` | Token-by-token LLM output | Chat interfaces |
| `debug` | Maximum detail | Deep debugging |

### Sync vs Async
- `stream()` - Synchronous, blocks while streaming
- `astream()` - Async, allows concurrent operations

### Stream Mode: Updates

`stream_mode="updates"` yields only the **changes** from each node. Each chunk is a dict mapping node name to the updates it produced. Good for progress indicators.

In [41]:
# Stream mode: "updates" - See changes from each node
print("Stream mode: updates")
print("="*50)

config = {"configurable": {"thread_id": "stream-demo-1"}}

for chunk in agent.stream(
    {"messages": [HumanMessage(content="What's the weather in London?")]},
    config=config,
    stream_mode="updates"
):
    # Each chunk shows which node ran and what it changed
    for node_name, updates in chunk.items():
        print(f"\n[{node_name}] returned:")
        if "messages" in updates:
            for msg in updates["messages"]:
                content = msg.content[:100] if msg.content else f"tool_calls={msg.tool_calls}"
                print(f"  {msg.type}: {content}")

Stream mode: updates

[llm_call] returned:
  ai: tool_calls=[{'name': 'get_weather', 'args': {'city': 'London'}, 'id': 'call_I8WkJHhsVD08mHPbPoCAcMzN', 'type': 'tool_call'}]

[tool_node] returned:
  tool: The weather in London is 72°F and sunny.

[llm_call] returned:
  ai: It's 72°F and sunny in London.


### Stream Mode: Values

`stream_mode="values"` yields the **full state** after each node. More verbose but useful for debugging when you need to see the complete picture at each step.

In [42]:
# Stream mode: "values" - Full state after each node
print("Stream mode: values")
print("="*50)

config = {"configurable": {"thread_id": "stream-demo-2"}}

for i, state in enumerate(agent.stream(
    {"messages": [HumanMessage(content="Add 5 and 3")]},
    config=config,
    stream_mode="values"
)):
    print(f"\n[State {i}] {len(state['messages'])} messages")
    print(f"  Latest: {state['messages'][-1].type}")

Stream mode: values

[State 0] 1 messages
  Latest: human

[State 1] 2 messages
  Latest: ai

[State 2] 3 messages
  Latest: tool

[State 3] 4 messages
  Latest: ai


### Stream Mode: Messages (Token-by-Token)

`stream_mode="messages"` yields LLM output **token by token** as `(chunk, metadata)` tuples. Perfect for chat UIs that show a typing indicator. The text appears progressively as the LLM generates it.

In [43]:
# Stream mode: "messages" - Token-by-token streaming
# Perfect for chat interfaces that show typing indicators

print("Stream mode: messages (token-by-token)")
print("="*50)

config = {"configurable": {"thread_id": "stream-demo-3"}}

for message_chunk, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me a very short joke.")]},
    config=config,
    stream_mode="messages"
):
    # Only print content tokens (not tool calls)
    if message_chunk.content:
        print(message_chunk.content, end="", flush=True)

print()  # Newline at end

Stream mode: messages (token-by-token)
I told my computer I needed a break — it froze.


---
# Summary

This notebook covered LangGraph's advanced features:

| Feature | Purpose | Key Classes |
|---------|---------|-------------|
| **Checkpointer** | Short-term memory (per-thread) | `InMemorySaver`, `PostgresSaver` |
| **Interrupts** | Human-in-the-loop | `interrupt()`, `Command(resume=...)` |
| **Store** | Long-term memory (cross-thread) | `InMemoryStore`, `PostgresStore` |
| **Semantic Search** | Vector search over memories | `store.search(query=...)` |
| **Streaming** | Real-time execution updates | `stream_mode="updates|values|messages"` |

### Production Considerations

- Use **database-backed** checkpointers and stores (Postgres, Redis) for persistence
- Set up **LangSmith** for tracing and debugging
- Use **`interrupt_before`/`interrupt_after`** for debugging without code changes
- Consider **message trimming** or **summarization** for long conversations

### Related Concepts Not Covered

- **Subgraphs**: Nest graphs for complex workflows
- **Middleware**: Add cross-cutting concerns (logging, caching, guardrails)
- **LangGraph Cloud**: Deploy and scale your agents